# Local release, tracking and API evidence

In [1]:
import json
import sys
from pathlib import Path

sys.path.insert(0, str(Path("src").resolve()))
root = Path.cwd()
release = json.loads((root / "models/release.json").read_text())
{k: release[k] for k in ["release_id", "mlflow_run_id", "validation_gate_passed", "sha256"]}

{'release_id': 'soil-screening-v1',
 'mlflow_run_id': 'b20d2cfbe7ad4ecd8cf6085f34396cdb',
 'validation_gate_passed': True,
 'sha256': '250f1f41e159815eb884e62f63460495631ac6aa10976b1e0d7ae4d07e04e25d'}

In [2]:
import mlflow
from mlflow.tracking import MlflowClient

mlflow.set_tracking_uri((root / "models/mlruns").resolve().as_uri())
client = MlflowClient(tracking_uri=(root / "models/mlruns").resolve().as_uri())
run = client.get_run(release["mlflow_run_id"])
assert run.info.status == "FINISHED"
assert client.list_artifacts(run.info.run_id, "evidence")
run.data.metrics

{'test_average_precision': 0.9508074484273081,
 'test_f1': 0.9302325581395349,
 'validation_f1': 0.968421052631579}

In [3]:
from agri_platform.api import load_release

verified, state = load_release(root)
monitor = json.loads((root / "reports/replay-monitoring.json").read_text())
assert monitor["total_accepted"] == 40
assert monitor["source_counts"] == {"historical-replay": 40}
{k: monitor[k] for k in ["total_accepted", "latency_p50_ms", "latency_p95_ms", "source_counts"]}

{'total_accepted': 40,
 'latency_p50_ms': 24.89895000053366,
 'latency_p95_ms': 26.115465000111726,
 'source_counts': {'historical-replay': 40}}

Historical replay is explicit. Model metrics repeat the fixed sensor study; PSI and local timings do not validate irrigation decisions or production service capacity.